In [ ]:
# Colab setup - run this first. Installs m3 plus the tutorial plotting extras.
# (Colab already ships PyTorch, which m3 uses as its engine.)
%pip install -q "git+https://github.com/PYangLab/M3.git" scanpy umap-learn

# Feature attribution

Once m3 can predict disease, attribution asks which genes, cells and cell types drove
that prediction. We train on the full reference, then trace each prediction back to
the genes, cells, and cell types behind it.

In [ ]:
import os
import pandas as pd
import m3

OUT = "./tutorial_out_demo/03_attribution"
os.makedirs(OUT, exist_ok=True)

## 1. Load the demo data

In [ ]:
data = m3.datasets.liu_demo()
print(data)

## 2. Build and train the model

Same setup as the patient-prediction tutorial, minus the held-out batch (we
attribute on the full reference). `.train()` fits both the integration model and the
disease predictor; attribution explains that predictor.

In [ ]:
model = m3.M3(
    data,
    condition_keys=["cond_group", "Age_interval"],
    target_condition="cond_group",
    celltype_key="mergedcelltype",
    batch_key="batch",
    donor_key="sample_id",
    embedding_dim=30,
)
model.train(max_epochs=500)
print("capabilities:", model.capabilities)

## 3. Attribute and rank the cell types

`model.attribute(reference_labels=["HC"])` scores how much each gene, cell, and cell
type pushed the prediction away from the reference label (HC, i.e. healthy). Here we
look at the **top cell types**: `attr.top_celltypes(...)` ranks them, keeping only
cell types with enough cells in each condition so a small group can't mislead the
ranking.

In [ ]:
attr = model.attribute(reference_labels=["HC"])
print("top cell types (filtered: >= 200 cells per condition):\n",
      attr.top_celltypes(min_cells_per_condition=200).to_string(index=False))

## 4. Top genes

`attr.top_genes(...)` ranks the genes by how strongly they drove the prediction.
`min_cells_per_condition` keeps only cell types with enough cells in each condition
(so a tiny group can't dominate the ranking), and housekeeping / ribosomal genes are
dropped.

In [ ]:
top_rna = attr.top_genes(n=100, min_cells_per_condition=200, modality="rna")
print(f"top-100 RNA genes (computed over {int(top_rna['n_celltypes_used'].iloc[0])} balanced cell types):")
print(top_rna.head(15).to_string(index=False))
top_rna.to_csv(os.path.join(OUT, "gene_importance_top100_rna.csv"), index=False)

## 5. Save the attribution

The full cells-by-features attribution matrix, as an HDF5 array and an AnnData.

In [ ]:
import h5py
import anndata as ad

attr_mat = attr.attribution
present = [m for m in ("rna", "adt", "atac") if m in data.modality_names]
feat_names = [f for mm in present for f in list(data.var[mm])][: attr_mat.shape[1]]

with h5py.File(os.path.join(OUT, "attribution.h5"), "w") as f:
    f["data"] = attr_mat

_obs = model.cell_metadata.astype(str).reset_index(drop=True)
ad.AnnData(X=attr_mat, obs=_obs, var=pd.DataFrame(index=feat_names)).write_h5ad(
    os.path.join(OUT, "attribution.h5ad"))
print("saved attribution -> .h5 (['data']) / .h5ad in", OUT)

**Done.** The genes, cells, and cell types behind m3's Severe-vs-HC prediction.